In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import os
import time

# Set plot styles
plt.style.use('seaborn-v0_8-whitegrid')  # Update this line
sns.set_palette('viridis')

In [14]:
def load_enhanced_dataset():
    """Load the enhanced dataset from the previous notebook"""
    # Path to the enhanced dataset
    enhanced_path = '../../data/processed/enhanced_features_dataset.csv'
    
    if os.path.exists(enhanced_path):
        print(f"Loading enhanced dataset from: {enhanced_path}")
        try:
            df = pd.read_csv(enhanced_path)
            print(f"Loaded enhanced dataset: {df.shape[0]} rows, {df.shape[1]} columns")
            return df
        except Exception as e:
            print(f"Error loading enhanced dataset: {e}")
            return None
    else:
        print(f"Enhanced dataset not found at {enhanced_path}. Please run composite_features.ipynb first.")
        return None

# Load the enhanced dataset
df_with_attack_features = load_enhanced_dataset()

# Display basic info if loaded successfully
if df_with_attack_features is not None:
    print("\nDataFrame loaded successfully. Basic info:")
    df_with_attack_features.info()
    
    # Display attack type distribution
    if 'attack_type' in df_with_attack_features.columns:
        print("\nAttack type distribution:")
        print(df_with_attack_features['attack_type'].value_counts())
else:
    print("\nDataFrame could not be loaded. Cannot proceed.")

Loading enhanced dataset from: ../../data/processed/enhanced_features_dataset.csv
Loaded enhanced dataset: 2000 rows, 90 columns

DataFrame loaded successfully. Basic info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 90 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Dst Port             2000 non-null   int64  
 1   Protocol             2000 non-null   int64  
 2   Timestamp            2000 non-null   object 
 3   Flow Duration        2000 non-null   float64
 4   Tot Fwd Pkts         2000 non-null   float64
 5   Tot Bwd Pkts         2000 non-null   float64
 6   TotLen Fwd Pkts      2000 non-null   float64
 7   TotLen Bwd Pkts      2000 non-null   float64
 8   Fwd Pkt Len Max      2000 non-null   float64
 9   Fwd Pkt Len Min      2000 non-null   float64
 10  Fwd Pkt Len Mean     2000 non-null   float64
 11  Fwd Pkt Len Std      2000 non-null   float64
 12  Bwd Pkt Len Max

In [15]:
def split_by_attack_type(df):
    """Split the dataset by attack type"""
    if df is None or 'attack_type' not in df.columns:
        print("Error: Cannot split dataset without 'attack_type' column")
        return {}
        
    datasets = {}
    # Get normal traffic
    datasets['Normal'] = df[df['attack_type'] == 'Normal']
    
    # Get each attack type
    for attack_type in df['attack_type'].unique():
        if attack_type != 'Normal':
            attack_samples = df[df['attack_type'] == attack_type]
            datasets[attack_type] = attack_samples
            
            # Check if we have enough samples for meaningful analysis
            if len(attack_samples) < 10:
                print(f"Warning: {attack_type} has only {len(attack_samples)} samples, which may be insufficient for reliable feature importance analysis")

In [16]:
def get_important_features_for_attack(normal_df, attack_df, all_features, top_n=15, min_samples=10):
    """Identify important features for detecting a specific attack type"""
    if len(normal_df) == 0:
        print("Error: No normal samples provided")
        return {'top_features': [], 'all_importances': []}
    
    if len(attack_df) < min_samples:
        print(f"Warning: Attack class has only {len(attack_df)} samples, which is below the minimum of {min_samples}")
        print("Using manual feature selection based on known attack patterns instead")
        
        # This is where we implement a fallback method when we don't have enough samples
        # We'll use domain knowledge from your Phase 1 documents
        
        # First, get only numeric features
        numeric_features = [col for col in all_features 
                          if col in attack_df.columns
                          and pd.api.types.is_numeric_dtype(attack_df[col])
                          and col not in ['is_attack', 'attack_type', 'Label', 'target', 'source']]
        
        # Calculate mean difference between attack and normal for each feature
        feature_diffs = []
        for feature in numeric_features:
            normal_mean = normal_df[feature].mean()
            attack_mean = attack_df[feature].mean()
            
            # Calculate percent difference if possible
            if normal_mean != 0:
                percent_diff = abs((attack_mean - normal_mean) / normal_mean)
            else:
                percent_diff = abs(attack_mean)
                
            feature_diffs.append((feature, percent_diff))
        
        # Sort by difference (largest first)
        feature_diffs.sort(key=lambda x: x[1], reverse=True)
        
        # Print top differences
        print("\nTop features by mean difference:")
        for i, (feature, diff) in enumerate(feature_diffs[:15]):
            normal_mean = normal_df[feature].mean()
            attack_mean = attack_df[feature].mean()
            print(f"{i+1}. {feature}: {diff:.4f} (Normal: {normal_mean:.4f}, Attack: {attack_mean:.4f})")
        
        # Return top features
        top_features = [feature for feature, _ in feature_diffs[:top_n]]
        
        return {
            'top_features': top_features,
            'all_importances': feature_diffs
        }
    
    try:
        # Standard approach with Random Forest when we have enough samples
        # Combine datasets
        normal_sample_size = min(len(normal_df), len(attack_df)*10)
        combined_df = pd.concat([
            normal_df.sample(normal_sample_size, random_state=42),
            attack_df
        ])
        
        # Create binary target
        combined_df['target'] = combined_df['attack_type'] != 'Normal'
        
        # Remove non-feature columns and ensure we only use numeric features
        exclude_cols = ['is_attack', 'attack_type', 'Label', 'target', 'source', 'Timestamp']
        feature_cols = [col for col in all_features 
                      if col not in exclude_cols 
                      and col in combined_df.columns
                      and pd.api.types.is_numeric_dtype(combined_df[col])]
        
        print(f"Using {len(feature_cols)} numeric features for importance analysis")
        
        # Use Random Forest for feature importance
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(combined_df[feature_cols], combined_df['target'])
        
        # Get feature importances
        importances = rf.feature_importances_
        
        # Create sorted list of features and importances
        feature_importance = [(feature, importance) 
                             for feature, importance in zip(feature_cols, importances)]
        feature_importance.sort(key=lambda x: x[1], reverse=True)
        
        # Print top features
        print("\nTop features by Random Forest importance:")
        for i, (feature, importance) in enumerate(feature_importance[:15]):
            print(f"{i+1}. {feature}: {importance:.6f}")
        
        # Get top N features
        top_features = [feature for feature, _ in feature_importance[:top_n]]
        
        return {
            'top_features': top_features,
            'all_importances': feature_importance
        }
    except Exception as e:
        print(f"Error in feature importance calculation: {e}")
        import traceback
        traceback.print_exc()
        return {'top_features': [], 'all_importances': []}

# Get important features for each attack type
if df_with_attack_features is not None and attack_datasets:
    attack_features = {}
    all_features = df_with_attack_features.columns.tolist()
    
    if 'Normal' not in attack_datasets:
        print("Error: No 'Normal' samples found in the dataset. Cannot perform attack analysis.")
    else:
        # Check if we have attack samples
        attack_types = [at for at in attack_datasets.keys() if at != 'Normal']
        if not attack_types:
            print("Warning: No attack samples found in the dataset. Adding known important features from Phase 1 documents.")
            
            # Fallback to known important features from Phase 1 documents
            attack_features['General'] = {
                'top_features': [
                    'Flow IAT Max', 'Flow Duration', 'Bwd Pkt Len Max', 'ACK Flag Cnt',
                    'Dst Port', 'TotLen Fwd Pkts', 'PSH Flag Cnt', 'Bwd IAT Max',
                    'Tot Fwd Pkts', 'Flow IAT Mean', 'Init Bwd Win Byts', 'Flow IAT Min',
                    'Fwd Pkt Len Max', 'RST Flag Cnt', 'Pkt Len Var',
                    # Add the composite features we created
                    'SQL_Injection_Score', 'Brute_Force_Score', 'XSS_Score',
                    'Pkt_Size_Ratio', 'IAT_Ratio', 'Flag_Combination', 'Pkts_Per_Second'
                ],
                'all_importances': [(f, 1.0) for f in ['Flow IAT Max', 'Flow Duration', 'Bwd Pkt Len Max', 'ACK Flag Cnt',
                                                      'Dst Port', 'TotLen Fwd Pkts', 'PSH Flag Cnt', 'Bwd IAT Max',
                                                      'Tot Fwd Pkts', 'Flow IAT Mean', 'Init Bwd Win Byts', 'Flow IAT Min',
                                                      'Fwd Pkt Len Max', 'RST Flag Cnt', 'Pkt Len Var',
                                                      'SQL_Injection_Score', 'Brute_Force_Score', 'XSS_Score',
                                                      'Pkt_Size_Ratio', 'IAT_Ratio', 'Flag_Combination', 'Pkts_Per_Second']]
            }
        else:
            # Process each attack type
            for attack_type in attack_types:
                print(f"\nAnalyzing features for {attack_type}...")
                result = get_important_features_for_attack(
                    attack_datasets['Normal'],
                    attack_datasets[attack_type],
                    all_features
                )
                if result['top_features']:
                    attack_features[attack_type] = result
else:
    attack_features = {}


Analyzing features for Brute Force-Web...
Using manual feature selection based on known attack patterns instead

Top features by mean difference:
1. RST Flag Cnt: 7.8062 (Normal: 0.0787, Attack: 0.6931)
2. ECE Flag Cnt: 7.8062 (Normal: 0.0787, Attack: 0.6931)
3. Flag_Combination: 2.4891 (Normal: 1.3679, Attack: 4.7726)
4. Fwd Pkt Len Std: 2.0873 (Normal: 1.8364, Attack: 5.6695)
5. SQL_Injection_Score: 2.0335 (Normal: 0.0006, Attack: 0.0017)
6. Fwd IAT Std: 1.9715 (Normal: 4.9533, Attack: 14.7189)
7. PSH Flag Cnt: 1.6941 (Normal: 0.3712, Attack: 1.0000)
8. Bwd IAT Min: 1.5191 (Normal: 2.8220, Attack: 7.1091)
9. Flow IAT Std: 1.5033 (Normal: 5.7688, Attack: 14.4409)
10. Pkt_Size_Ratio: 1.0767 (Normal: 2.4946, Attack: 5.1807)
11. Fwd Pkt Len Min: 1.0000 (Normal: 1.1238, Attack: 0.0000)
12. Bwd Pkt Len Min: 1.0000 (Normal: 1.3232, Attack: 0.0000)
13. Fwd PSH Flags: 1.0000 (Normal: 0.0260, Attack: 0.0000)
14. Pkt Len Min: 1.0000 (Normal: 1.1210, Attack: 0.0000)
15. FIN Flag Cnt: 1.0000 (No

In [17]:
def create_feature_union(attack_features):
    """Create a union of important features across attack types"""
    if not attack_features:
        print("No attack features to union")
        return []
        
    all_important_features = set()
    for attack_type, result in attack_features.items():
        all_important_features.update(result['top_features'])
    return list(all_important_features)

# Get union of important features
if 'attack_features' in locals() and attack_features:
    union_features = create_feature_union(attack_features)
    if union_features:
        print(f"\nUnion of important features across attack types ({len(union_features)} features):")
        for feature in union_features:
            print(f"- {feature}")
    else:
        print("\nNo features were found across attack types. Using default features.")
        # Use default features from Phase 1
        union_features = [
            'Flow IAT Max', 'Flow Duration', 'Bwd Pkt Len Max', 'ACK Flag Cnt',
            'Dst Port', 'TotLen Fwd Pkts', 'PSH Flag Cnt', 'Bwd IAT Max',
            'Tot Fwd Pkts', 'Flow IAT Mean', 'Init Bwd Win Byts', 'Flow IAT Min',
            'Fwd Pkt Len Max', 'RST Flag Cnt', 'Pkt Len Var',
            # Add composite features
            'SQL_Injection_Score', 'Brute_Force_Score', 'XSS_Score',
            'Pkt_Size_Ratio', 'IAT_Ratio', 'Flag_Combination', 'Pkts_Per_Second'
        ]
        print(f"Using {len(union_features)} default features")
else:
    union_features = []
    print("No attack features available for union")


Union of important features across attack types (15 features):
- Bwd Pkt Len Min
- FIN Flag Cnt
- Bwd IAT Min
- ECE Flag Cnt
- Fwd Pkt Len Min
- RST Flag Cnt
- SQL_Injection_Score
- PSH Flag Cnt
- Flow IAT Std
- Fwd IAT Std
- Flag_Combination
- Pkt_Size_Ratio
- Pkt Len Min
- Fwd Pkt Len Std
- Fwd PSH Flags


In [18]:
def remove_redundant_features(df, features, correlation_threshold=0.9):
    """Remove redundant features based on correlation"""
    if df is None or not features:
        print("Error: DataFrame or features list is empty")
        return []
        
    try:
        # Select features that exist in the dataframe
        available_features = [f for f in features if f in df.columns]
        if len(available_features) < len(features):
            missing = set(features) - set(available_features)
            print(f"Warning: {len(missing)} features not found in DataFrame: {missing}")
        
        if len(available_features) < 2:
            print("Not enough features to analyze correlations")
            return available_features
            
        # Calculate correlation matrix
        df_subset = df[available_features]
        corr_matrix = df_subset.corr().abs()
        
        # Create upper triangle mask
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        
        # Find features with correlation higher than threshold
        to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
        
        # Get non-redundant features
        non_redundant = [f for f in available_features if f not in to_drop]
        
        print(f"Removing {len(to_drop)} redundant features with correlation > {correlation_threshold}")
        if to_drop:
            print(f"Redundant features: {to_drop}")
        print(f"Remaining features: {len(non_redundant)}")
        
        return non_redundant
    except Exception as e:
        print(f"Error removing redundant features: {e}")
        return features  # Return original features if error occurs

# Save features to files by attack type
def save_attack_specific_features(attack_features):
    """Save features for each attack type to separate files"""
    model_dir = '../../src/models'
    os.makedirs(model_dir, exist_ok=True)
    
    for attack_type, result in attack_features.items():
        filename = f"{model_dir}/{attack_type.replace(' ', '_').lower()}_features.txt"
        with open(filename, 'w') as f:
            for feature in result['top_features']:
                f.write(f"{feature}\n")
        print(f"Saved features for {attack_type} to {filename}")

# Remove redundant features and save results
if df_with_attack_features is not None and 'union_features' in locals() and union_features:
    # Remove redundant features
    final_features = remove_redundant_features(df_with_attack_features, union_features)
    
    # Ensure the directory exists
    os.makedirs('../../src/models', exist_ok=True)
    
    # Save final feature list
    try:
        with open('../../src/models/selected_features.txt', 'w') as f:
            for feature in final_features:
                f.write(f"{feature}\n")
        print("\nFinal feature list saved to src/models/selected_features.txt")
        
        # Also save attack-specific features
        if 'attack_features' in locals() and attack_features:
            save_attack_specific_features(attack_features)
    except Exception as e:
        print(f"Error saving feature lists: {e}")
else:
    print("Cannot determine final features - required data not available")

Removing 4 redundant features with correlation > 0.9
Redundant features: ['Fwd Pkt Len Min', 'RST Flag Cnt', 'SQL_Injection_Score', 'Pkt Len Min']
Remaining features: 11

Final feature list saved to src/models/selected_features.txt
Saved features for Brute Force-Web to ../../src/models/brute_force-web_features.txt
